In [1]:
!pip install ultralytics pyttsx3 opencv-python
!apt-get install -y espeak ffmpeg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 31.1 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
The following additional packages will be installed:
  espeak-data libespeak1 libportaudio2 libsonic0
The following NEW packages will be installed:
  espeak espeak-data libespeak1 libportaudio2 libsonic0
0 upgraded, 5 newly installed, 0 to remove and 2 not upgraded.
Need to get 1,382 kB of archives.
After this operation, 3,178 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libportaudio2 amd64 19.6.0-1.1 [65.3 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libsonic0 amd64 0.2.0-11build1 [10.3 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 espeak-data amd64 1.48.15+dfsg-3 [1,085 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libespeak1 amd64 1.48.15+dfs

In [5]:
from google.colab import files

uploaded = files.upload()

Saving cristhmas_video.mp4 to cristhmas_video.mp4


In [6]:
import cv2
import pyttsx3
from ultralytics import YOLO
from google.colab import files
from collections import defaultdict
import time
import os

In [7]:
# -------------------------
# Load YOLOv8n model
# -------------------------
model = YOLO("yolov8n.pt")

# -------------------------
# Video path
# -------------------------
video_path = list(uploaded.keys())[0]

cap = cv2.VideoCapture(video_path)

fps = int(cap.get(cv2.CAP_PROP_FPS))
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print("FPS:", fps)
print("Frames:", frame_count)


FPS: 29
Frames: 500


In [8]:
import pyttsx3
from collections import defaultdict
import time
import os

engine = None
try:
    engine = pyttsx3.init()
    # Attempt to set a more common English voice if initialization succeeds
    voices = engine.getProperty('voices')
    english_voice_id = None
    for voice in voices:
        if 'en' in voice.id.lower(): # Search for any English voice
            english_voice_id = voice.id
            break
    if english_voice_id:
        engine.setProperty('voice', english_voice_id)
        print(f"pyttsx3 initialized with voice: {english_voice_id}")
    else:
        print("Warning: No specific English voice found, using pyttsx3's default.")

except Exception as e:
    print(f"Error initializing pyttsx3 engine: {e}")
    print("Text-to-speech functionality will be unavailable.")
    # Create a dummy engine so the rest of the code doesn't break
    class DummyEngine:
        def setProperty(self, *args, **kwargs): pass
        def say(self, *args, **kwargs): pass
        def runAndWait(self): pass
        def stop(self): pass
    engine = DummyEngine()

# Continue with setting properties if a real engine was initialized
if not isinstance(engine, DummyEngine):
    engine.setProperty('rate', 150)
    engine.setProperty('volume', 1.0)
else:
    print("Skipping engine property settings due to initialization failure.")


# -------------------------
# Cooldown tracker
# -------------------------
last_spoken_time = defaultdict(lambda: -10)

# -------------------------
# Save descriptions
# -------------------------
all_descriptions = []

# -------------------------
# Helper functions
# -------------------------
def get_position(x_center, frame_width):

    if x_center < frame_width / 3:
        return "LEFT"

    elif x_center < 2 * frame_width / 3:
        return "CENTRE"

    else:
        return "RIGHT"

Error initializing pyttsx3 engine: SetVoiceByName failed with unknown return code -1 for voice: gmw/en
Text-to-speech functionality will be unavailable.
Skipping engine property settings due to initialization failure.


In [9]:
def get_distance(box_area, frame_area):

    ratio = box_area / frame_area

    if ratio > 0.15:
        return "NEAR"

    elif ratio > 0.05:
        return "MID"

    else:
        return "FAR"


# -------------------------
# Process video
# -------------------------
frame_index = 0
current_second = 0

while cap.isOpened():

    ret, frame = cap.read()

    if not ret:
        break

    frame_index += 1

    # Process once per second
    if frame_index % fps != 0:
        continue

    current_second += 1

    h, w, _ = frame.shape
    frame_area = w * h

    results = model(frame)

    descriptions = []

    for result in results:

        boxes = result.boxes

        for box in boxes:

            confidence = float(box.conf[0])

            # confidence > 0.5
            if confidence < 0.5:
                continue

            cls_id = int(box.cls[0])
            class_name = model.names[cls_id]

            x1, y1, x2, y2 = map(int, box.xyxy[0])

            # Bounding box center
            x_center = (x1 + x2) / 2

            # Bounding box area
            box_area = (x2 - x1) * (y2 - y1)

            # LEFT / CENTRE / RIGHT
            position = get_position(x_center, w)

            # NEAR / MID / FAR
            distance = get_distance(box_area, frame_area)

            # -------------------------
            # Cooldown: 2 seconds
            # -------------------------
            if current_second - last_spoken_time[class_name] < 2:
                continue

            description = (
                f"{class_name} on the "
                f"{position}, {distance.lower()}."
            )

            descriptions.append(description)

            last_spoken_time[class_name] = current_second

    # Speak every second
    if descriptions:

        final_text = " ".join(descriptions)

        print(f"[{current_second}s] {final_text}")

        all_descriptions.append(final_text)

        engine.say(final_text)
        engine.runAndWait()

cap.release()

print("\nFinished Processing")


0: 384x640 1 potted plant, 390.2ms
Speed: 16.7ms preprocess, 390.2ms inference, 41.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 204.4ms
Speed: 6.3ms preprocess, 204.4ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 138.0ms
Speed: 2.5ms preprocess, 138.0ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 135.7ms
Speed: 5.0ms preprocess, 135.7ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 148.1ms
Speed: 4.0ms preprocess, 148.1ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 156.6ms
Speed: 4.2ms preprocess, 156.6ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 147.5ms
Speed: 4.0ms preprocess, 147.5ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 teddy bear, 164.6ms
Speed: 3.2ms preproc

In [10]:
!pip install gtts
from gtts import gTTS

# Combine all generated descriptions
full_text = " ".join(all_descriptions)

print(full_text)

if full_text.strip():

    audio_file = "scene_description.mp3"

    tts = gTTS(text=full_text, lang='en')

    tts.save(audio_file)

    print("Audio saved:", audio_file)

else:
    print("No descriptions generated")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 4.7 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.4.0
    Uninstalling click-8.4.0:
      Successfully uninstalled click-8.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
wandb 0.27.0 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.
typer 0.25.1 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.

No descriptions generated


In [13]:
from google.colab import files

files.download("/content/cristhmas_video.mp4")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
import cv2
from ultralytics import YOLO
from collections import defaultdict

model = YOLO("yolov8n.pt")

video_path = "cristhmas_video.mp4"

cap = cv2.VideoCapture(video_path)

fps = int(cap.get(cv2.CAP_PROP_FPS))

print("FPS:", fps)

last_spoken_time = defaultdict(lambda: -10)
all_descriptions = []

frame_index = 0
current_second = 0


def get_position(x_center, frame_width):

    if x_center < frame_width / 3:
        return "LEFT"

    elif x_center < 2 * frame_width / 3:
        return "CENTRE"

    return "RIGHT"


def get_distance(box_area, frame_area):

    ratio = box_area / frame_area

    if ratio > 0.15:
        return "NEAR"

    elif ratio > 0.03:
        return "MID"

    return "FAR"


while cap.isOpened():

    ret, frame = cap.read()

    if not ret:
        break

    frame_index += 1

    # every second
    if frame_index % fps != 0:
        continue

    current_second += 1

    h, w, _ = frame.shape
    frame_area = h * w

    results = model(frame)

    descriptions = []

    print(f"\nSecond {current_second}")

    for result in results:

        for box in result.boxes:

            confidence = float(box.conf[0])

            # Lowered threshold for testing
            if confidence < 0.3:
                continue

            cls_id = int(box.cls[0])
            class_name = model.names[cls_id]

            x1, y1, x2, y2 = map(int, box.xyxy[0])

            x_center = (x1 + x2) / 2
            box_area = (x2 - x1) * (y2 - y1)

            position = get_position(x_center, w)
            distance = get_distance(box_area, frame_area)

            print(
                f"Detected: {class_name} "
                f"({confidence:.2f})"
            )

            # cooldown
            if current_second - last_spoken_time[class_name] < 2:
                continue

            sentence = (
                f"{class_name} on the "
                f"{position}, {distance.lower()}."
            )

            descriptions.append(sentence)

            last_spoken_time[class_name] = current_second

    if descriptions:

        final_text = " ".join(descriptions)

        print("Description:", final_text)

        all_descriptions.append(final_text)

cap.release()

print("\nDescriptions generated:")
print(all_descriptions)

FPS: 29

0: 384x640 1 potted plant, 134.2ms
Speed: 3.4ms preprocess, 134.2ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

Second 1

0: 384x640 (no detections), 135.3ms
Speed: 3.9ms preprocess, 135.3ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

Second 2

0: 384x640 (no detections), 138.6ms
Speed: 3.8ms preprocess, 138.6ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

Second 3

0: 384x640 (no detections), 152.7ms
Speed: 4.6ms preprocess, 152.7ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

Second 4

0: 384x640 (no detections), 146.7ms
Speed: 4.5ms preprocess, 146.7ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

Second 5

0: 384x640 (no detections), 148.8ms
Speed: 4.0ms preprocess, 148.8ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

Second 6

0: 384x640 (no detections), 127.6ms
Speed: 3.5ms preprocess, 127.6ms inference, 0.7ms postprocess per image at shape (1, 3

In [15]:
from gtts import gTTS
from google.colab import files

full_text = " ".join(all_descriptions)

tts = gTTS(full_text)
tts.save("scene_description.mp3")

files.download("scene_description.mp3")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>